# Global Tourism Intelligence
## Notebook 01: Data Exploration

### Project Overview
An end-to-end data engineering project that analyzes
global tourism trends, international tourist arrivals,
tourism receipts, and accommodation statistics.

### Objectives
- Collect real-world tourism data.
- Explore and understand source datasets.
- Identify missing values and data quality issues.
- Understand country-wise and yearly tourism trends.
- Prepare data for the Bronze, Silver, and Gold layers.

### Primary Data Sources
1. World Bank Open Data
2. UN Tourism
3. Eurostat

### Technology Stack
- Python
- Apache Spark (PySpark)
- Databricks
- Delta Lake
- Power BI


In [0]:
import requests
import pandas as pd
import json
from datetime import datetime

print("Libraries imported successfully!")

## World Bank API connection

In [0]:
import requests
import pandas as pd
import time

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configure retry mechanism
session = requests.Session()

retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)

# World Bank API
BASE_URL = "https://api.worldbank.org/v2"

INDICATOR = "ST.INT.ARVL"

URL = f"{BASE_URL}/country/all/indicator/{INDICATOR}"

params = {
    "format": "json",
    "per_page": 1000,
    "page": 1
}

response = session.get(
    URL,
    params=params,
    timeout=(15, 120)
)

response.raise_for_status()

api_response = response.json()

metadata = api_response[0]

print("API Status:", response.status_code)
print("Total Records:", metadata["total"])
print("Total Pages:", metadata["pages"])

## Extract actual tourism records

In [0]:

import requests
import pandas as pd
import time

# Initialize extraction
records = []
total_pages = metadata["pages"]
total_expected = metadata["total"]

# Create HTTP session
session = requests.Session()

# Extract all pages
for page in range(1, total_pages + 1):

    params["page"] = page

    page_success = False

    # Retry failed requests up to 8 times
    for attempt in range(1, 9):

        try:
            response = session.get(
                URL,
                params=params,
                timeout=(15, 120)
            )

            response.raise_for_status()

            page_data = response.json()[1] or []

            page_records = []

            # Process records
            for item in page_data:

                country = item.get("country") or {}
                region = country.get("region") or {}
                income = country.get("incomeLevel") or {}

                page_records.append({

                    "country": country.get("value"),

                    "country_code": item.get("countryiso3code"),

                    "country_id": country.get("id"),

                    "region": region.get("value"),

                    "income_level": income.get("value"),

                    "year": int(item["date"]),

                    "indicator": (
                        item.get("indicator") or {}
                    ).get("id"),

                    "value": item.get("value"),

                    "unit": item.get("unit"),

                    "observation_status": item.get("obs_status")
                })

            # Add records only after successful page extraction
            records.extend(page_records)

            page_success = True

            print(
                f"Page {page}/{total_pages} completed | "
                f"Records collected: {len(records)}"
            )

            break

        except requests.exceptions.RequestException as e:

            wait_time = min(2 ** attempt, 60)

            print(
                f"Page {page} failed | "
                f"Attempt {attempt}/8 | "
                f"Error: {e}"
            )

            if attempt < 8:
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)

    # Stop if a page fails all retries
    if not page_success:
        raise RuntimeError(
            f"Extraction stopped at page {page}. "
            f"Collected {len(records)} records. "
            f"Rerun using the recovery code to resume."
        )

    time.sleep(1)

# Convert records into DataFrame
tourism_df = pd.DataFrame(records)

# Final extraction summary
print("\n========== EXTRACTION SUMMARY ==========")

print("Expected Records:", total_expected)
print("Extracted Records:", len(tourism_df))
print("Total Pages:", total_pages)

print(
    "Extraction Complete:",
    len(tourism_df) == total_expected
)

display(tourism_df.head(20))


## Dataset overview

In [0]:
print("Dataset Shape:", tourism_df.shape)

print("\nColumn Names:")
print(tourism_df.columns.tolist())

print("\nData Types:")
print(tourism_df.dtypes)

print("\nDataset Information:")
tourism_df.info()